## Unimodal - Clinical Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Installation

!pip install -q monai torch torchvision torchaudio lifelines torchtuples pycox


In [ ]:
!pip install --upgrade scikit-learn scikit-survival skorch


### Model 1. Clinical data (Unimodal - Coxnet)

In [ ]:
df_clinical = pd.read_csv("/content/drive/MyDrive/NSCLC/df_clinical.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler 
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.linear_model import CoxnetSurvivalAnalysis


In [ ]:
# # Model 1 - Unimodal Coxnet


# Data Loading and Preparation

df_clinical_m1 = pd.read_csv("/content/drive/MyDrive/NSCLC/df_clinical.csv")

clinical_cols = [
    "age", "clinical.T.Stage", "Clinical.N.Stage",
    "Clinical.M.Stage", "Overall.Stage",
    "gender_male", "Histology_large cell",
    "Histology_nos", "Histology_squamous cell carcinoma"
]

X = df_clinical_m1[clinical_cols].values.astype("float32")
y_time = df_clinical_m1["Survival.time"].values.astype("float32")
y_event = df_clinical_m1["deadstatus.event"].values.astype("int")

# Stratified cross-validation

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

cindex_folds = []
ibs_folds = []
risk_all, time_all, event_all = [], [], []

param_grid = {
    'l1_ratio': [0.1, 0.5, 0.9],
    'alpha_min_ratio': [0.01, 0.05, 0.1]
}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_event), 1):
    print(f"\n===== Fold {fold}/{N_SPLITS} (Coxnet) =====")

    # Split
    X_train_full, X_test = X[train_idx], X[test_idx]
    y_time_train_full, y_time_test = y_time[train_idx], y_time[test_idx]
    y_event_train_full, y_event_test = y_event[train_idx], y_event[test_idx]

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_train_full = imputer.fit_transform(X_train_full)
    X_train_full = scaler.fit_transform(X_train_full)

    X_test = imputer.transform(X_test)
    X_test = scaler.transform(X_test)

 
    y_train_struct = Surv.from_arrays(event=y_event_train_full.astype(bool), time=y_time_train_full)
    y_test_struct  = Surv.from_arrays(event=y_event_test.astype(bool), time=y_time_test)

    # Base model

    model_base = CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.01, fit_baseline_model=True)

    # Grid Search
    gcv = GridSearchCV(model_base, param_grid, cv=3, n_jobs=-1)
    gcv.fit(X_train_full, y_train_struct)

    best_model = gcv.best_estimator_
    print(f"Best settings: {gcv.best_params_}")

    # Metrics

    risk_scores = best_model.predict(X_test)
    cindex = concordance_index_censored(y_event_test.astype(bool), y_time_test, risk_scores)[0]
    cindex_folds.append(cindex)

    # IBS Calculation

    surv_funcs = best_model.predict_survival_function(X_test)
    t_min = max(y_time_test.min(), y_time_train_full.min())
    t_max = min(y_time_test.max(), y_time_train_full.max())
    grid_fold = np.linspace(t_min + 1, t_max - 1, 100)

    preds_surv = np.vstack([fn(grid_fold) for fn in surv_funcs])

    try:
        ibs_val = integrated_brier_score(y_train_struct, y_test_struct, preds_surv, grid_fold)
        ibs_folds.append(ibs_val)
    except:
        ibs_val = np.nan

    print(f"C-Index: {cindex:.4f} | IBS: {ibs_val:.4f}")

    risk_all.append(risk_scores)
    time_all.append(y_time_test)
    event_all.append(y_event_test)

# Results and Plotting

def get_stats(data):
    arr = np.array([x for x in data if not np.isnan(x)])
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    sem = stats.sem(arr)
    ci = stats.t.interval(0.95, len(arr)-1, loc=mean, scale=sem)
    return mean, std, ci

c_m, c_s, c_ci = get_stats(cindex_folds)
i_m, i_s, i_ci = get_stats(ibs_folds)

print("\n" + "="*60)
print("FINAL RESULTS - MODEL 1 (COXNET)")
print(f"C-Index: {c_m:.4f} ± {c_s:.4f} | IC 95%: [{c_ci[0]:.4f}, {c_ci[1]:.4f}]")
print(f"IBS:     {i_m:.4f} ± {i_s:.4f} | IC 95%: [{i_ci[0]:.4f}, {i_ci[1]:.4f}]")
print("="*60)

risks, times, events = np.concatenate(risk_all), np.concatenate(time_all), np.concatenate(event_all)
threshold = np.median(risks)
high, low = (risks >= threshold), (risks < threshold)
p_value = logrank_test(times[high], times[low], events[high], events[low]).p_value

plt.figure(figsize=(8, 6))
kmf = KaplanMeierFitter()
kmf.fit(times[high]/365.25, events[high], label="High Risk")
kmf.plot_survival_function()
kmf.fit(times[low]/365.25, events[low], label="Low Risk")
kmf.plot_survival_function()

plt.text(0.60, 0.35, f"p = {p_value:.4f}", transform=plt.gca().transAxes,
         fontsize=10, bbox=dict(boxstyle="round", facecolor="white", alpha=0.6))


plt.xlabel("Time (years)"); plt.ylabel("Survival Probability"); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("unimodal_model_1.png", dpi=300)
plt.show()

### Model 2. Clinical data (Unimodal - Random Survival Forest (RSF))

In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer

from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, integrated_brier_score

In [ ]:

# Output Settings

SAVE_DIR = "/content/drive/MyDrive/NSCLC/Resultados_Modelo_2/"
os.makedirs(SAVE_DIR, exist_ok=True)

# Data Loading and Preparation

df_clinical = pd.read_csv("/content/drive/MyDrive/NSCLC/df_clinical.csv")

clinical_cols = [
    "age", "clinical.T.Stage", "Clinical.N.Stage",
    "Clinical.M.Stage", "Overall.Stage",
    "gender_male", "Histology_large cell",
    "Histology_nos", "Histology_squamous cell carcinoma"
]

X = df_clinical[clinical_cols].values.astype("float32")
y_time = df_clinical["Survival.time"].values.astype("float32")
y_event = df_clinical["deadstatus.event"].values.astype("int")

# Cross-validation with grid search

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

cindex_folds = []
ibs_folds = []
risk_all, time_all, event_all = [], [], []

param_grid = {
    'n_estimators': [200, 300],            
    'max_features': ['sqrt', None],        
    'min_samples_leaf': [5, 10, 15],       
    'min_samples_split': [10, 20],         
    'max_depth': [None, 7, 10]            
}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_event), 1):
    print(f"\n===== Fold {fold}/{N_SPLITS} ====")

    X_train_full, X_test = X[train_idx], X[test_idx]
    y_time_train_full, y_time_test = y_time[train_idx], y_time[test_idx]
    y_event_train_full, y_event_test = y_event[train_idx], y_event[test_idx]

    imputer = SimpleImputer(strategy="median")
    X_train_full = imputer.fit_transform(X_train_full)
    X_test = imputer.transform(X_test)

    y_train_struct = Surv.from_arrays(event=y_event_train_full.astype(bool), time=y_time_train_full)
    y_test_struct  = Surv.from_arrays(event=y_event_test.astype(bool), time=y_time_test)

    # Initializes the base RSF model
    rsf_base = RandomSurvivalForest(n_jobs=-1, random_state=42)

    
    gcv = GridSearchCV(rsf_base, param_grid, cv=3, n_jobs=-1, error_score='raise')
    gcv.fit(X_train_full, y_train_struct)

    best_model = gcv.best_estimator_
    print(f"Melhores parâmetros no Fold {fold}: {gcv.best_params_}")

    # C-INDEX
    risk_scores = best_model.predict(X_test)
    cindex = concordance_index_censored(y_event_test.astype(bool), y_time_test, risk_scores)[0]
    cindex_folds.append(cindex)

    # IBS
    surv_funcs = best_model.predict_survival_function(X_test)
    t_min = max(y_time_test.min(), y_time_train_full.min())
    t_max = min(y_time_test.max(), y_time_train_full.max())
    grid_fold = np.linspace(t_min + 1, t_max - 1, 100)

    preds_surv = np.vstack([fn(grid_fold) for fn in surv_funcs])

    try:
        ibs_val = integrated_brier_score(y_train_struct, y_test_struct, preds_surv, grid_fold)
        ibs_folds.append(ibs_val)
    except:
        ibs_val = np.nan # Caso o grid falhe

    print(f"C-Index: {cindex:.4f} | IBS: {ibs_val:.4f}")

    risk_all.append(risk_scores)
    time_all.append(y_time_test)
    event_all.append(y_event_test)

# Final Statistical Report

def get_stats(data):
    arr = np.array([x for x in data if not np.isnan(x)])
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    sem = stats.sem(arr)
    ci = stats.t.interval(0.95, len(arr)-1, loc=mean, scale=sem)
    return mean, std, ci

c_m, c_s, c_ci = get_stats(cindex_folds)
i_m, i_s, i_ci = get_stats(ibs_folds)

risks, times, events = np.concatenate(risk_all), np.concatenate(time_all), np.concatenate(event_all)
threshold = np.median(risks)
high, low = (risks >= threshold), (risks < threshold)
p_value = logrank_test(times[high], times[low], events[high], events[low]).p_value

print("\n" + "="*60)
print(f"C-Index: {c_m:.4f} ± {c_s:.4f} | IC 95%: [{c_ci[0]:.4f}, {c_ci[1]:.4f}]")
print(f"IBS:     {i_m:.4f} ± {i_s:.4f} | IC 95%: [{i_ci[0]:.4f}, {i_ci[1]:.4f}]")
print(f"p-value: {p_value:.6f}")
print("="*60)


df_results = pd.DataFrame({
    "fold": range(1, N_SPLITS+1),
    "cindex": cindex_folds,
    "ibs": ibs_folds
})
df_results.to_csv(os.path.join(SAVE_DIR, "metrics_per_fold_model2.csv"), index=False)

# JSON 
import json 
summary = {
    "cindex_mean": float(c_m),
    "cindex_std": float(c_s),
    "cindex_ci_low": float(c_ci[0]),
    "cindex_ci_high": float(c_ci[1]),
    "ibs_mean": float(i_m),
    "ibs_std": float(i_s),
    "ibs_ci_low": float(i_ci[0]),
    "ibs_ci_high": float(i_ci[1]),
    "logrank_p_value": float(p_value)
}

with open(os.path.join(SAVE_DIR, "summary_model2.json"), "w") as f:
    json.dump(summary, f, indent=4)



plt.figure(figsize=(8, 6))
kmf = KaplanMeierFitter()
kmf.fit(times[high]/365.25, events[high], label=f"High Risk (n={sum(high)})") 
kmf.plot_survival_function(ci_show=True)
kmf.fit(times[low]/365.25, events[low], label=f"Low Risk (n={sum(low)})")
kmf.plot_survival_function(ci_show=True)

#plt.text(0.60, 0.35, f"p = {p_value:.4f}", transform=plt.gca().transAxes,
#         fontsize=10, bbox=dict(boxstyle="round", facecolor="white", alpha=0.6))

plt.xlabel("Time (years)"); plt.ylabel("Survival Probability"); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "km_model_2.png"), dpi=300)
plt.show()

## Model 3. Unimodal Image 

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import json

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r3d_18, R3D_18_Weights

from monai.transforms import Compose, EnsureTyped, RandFlipd, RandRotate90d, RandAffined, RandGaussianNoised
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter

from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.linear_model.coxph import BreslowEstimator
from sklearn.model_selection import StratifiedKFold, train_test_split

In [ ]:
# Model 3. ResNet 3D 18

# Config

CONFIG = {
    "BASE_PATH": "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/",
    "CLINICAL_CSV": "/content/drive/MyDrive/NSCLC/df_clinical.csv",
    "NUM_WORKERS": 2,
    "BATCH_SIZE_TRAIN": 8,
    "BATCH_SIZE_VAL": 2,
    "LR_HEAD": 5e-5,
    "EPOCHS": 150,
    "K_FOLDS": 5,
    "PATIENCE": 15,
    "WARMUP_EPOCHS": 10,
    "SEED": 42,
}

OUTPUT_DIR = "/content/drive/MyDrive/NSCLC/Resultados_Modelo_3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# SEED

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset

class LungDatasetImageOnly(Dataset):
    def __init__(self, df, base_path, augment=False):
        self.df = df.reset_index(drop=True)
        self.base_path = base_path
        self.transforms = Compose([
            EnsureTyped(keys=["vol"]),
            RandFlipd(keys=["vol"], prob=0.3, spatial_axis=(0,1)),
            RandRotate90d(keys=["vol"], prob=0.3),
            RandAffined(keys=["vol"], prob=0.3,
                        rotate_range=(0.15,0.15,0.15),
                        scale_range=(0.1,0.1,0.1),
                        translate_range=(6,6,6),
                        padding_mode="border"),
            RandGaussianNoised(keys=["vol"], prob=0.3, std=0.01),
        ]) if augment else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        base = os.path.join(self.base_path, row["PatientID"], "processed_final")
        ct = np.load(os.path.join(base, "ct_multi.npy")).astype(np.float32)
        seg = np.load(os.path.join(base, "seg.npy")).astype(np.float32)

        ct = np.transpose(ct, (3,0,1,2))
        seg = seg[np.newaxis, ...]
        vol = np.concatenate([ct, seg], axis=0)
        
        vol = np.ascontiguousarray(vol)

        data = {"vol": torch.from_numpy(vol)}
        if self.transforms:
            data = self.transforms(data)

        return (
            data["vol"].float(),
            torch.tensor(row["Survival.time"], dtype=torch.float32),
            torch.tensor(row["deadstatus.event"], dtype=torch.float32)
        )


class ImageOnlySurv(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = r3d_18(weights=R3D_18_Weights.DEFAULT)
        self.backbone.fc = nn.Identity()
        self.backbone.stem[0] = nn.Conv3d(3, 64, kernel_size=(3,7,7),
                                          stride=(1,2,2), padding=(1,3,3), bias=False)
        self.head = nn.Linear(512, 1)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)

def cox_loss(risk, time, event):
    # Garante estabilidade numérica
    risk = torch.clamp(risk, -50, 50)
    order = torch.argsort(time, descending=True)
    risk, event = risk[order], event[order]
    log_cumsum = torch.logcumsumexp(risk, dim=0)
    return -torch.mean((risk - log_cumsum) * event)


def validate(model, loader):
    model.eval()
    risks, times, events = [], [], []
    with torch.no_grad():
        for vol, time, event in loader:
            vol = vol.to(DEVICE)
            risks.extend(model(vol).cpu().numpy())
            times.extend(time.numpy())
            events.extend(event.numpy())
    return np.array(risks), np.array(times), np.array(events)

def get_stats(data):
    arr = np.array(data)
    mean = arr.mean()
    std = arr.std(ddof=1)
    ci = stats.t.interval(0.95, len(arr)-1, loc=mean, scale=stats.sem(arr))
    return mean, std, ci


def main():
    df = pd.read_csv(CONFIG["CLINICAL_CSV"])


    print(f"Total number of patients in the dataframe: {len(df)}")
    print(f"Total deaths (events): {int(df['deadstatus.event'].sum())}")
    print("-" * 30)

    skf = StratifiedKFold(n_splits=CONFIG["K_FOLDS"], shuffle=True, random_state=CONFIG["SEED"])

    
    cindex_folds, ibs_folds = [], []
    all_risks, all_times, all_events = [], [], []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df["deadstatus.event"]), 1):
        df_train_full = df.iloc[train_idx]
        df_test = df.iloc[test_idx]

        df_train, df_val = train_test_split(
            df_train_full, test_size=0.2,
            stratify=df_train_full["deadstatus.event"],
            random_state=CONFIG["SEED"]
        )

        print(f"\n===== FOLD {fold} =====")
        print(f"Train: N={len(df_train)} (Deaths: {int(df_train['deadstatus.event'].sum())})")
        print(f"Validation: N={len(df_val)}   (Deaths: {int(df_val['deadstatus.event'].sum())})")
        print(f"Test: N={len(df_test)}  (Deaths: {int(df_test['deadstatus.event'].sum())})")

        train_ld = DataLoader(LungDatasetImageOnly(df_train, CONFIG["BASE_PATH"], True),
                              batch_size=CONFIG["BATCH_SIZE_TRAIN"], shuffle=True, num_workers=CONFIG["NUM_WORKERS"])
        val_ld = DataLoader(LungDatasetImageOnly(df_val, CONFIG["BASE_PATH"], False),
                            batch_size=CONFIG["BATCH_SIZE_VAL"], shuffle=False, num_workers=CONFIG["NUM_WORKERS"])
        test_ld = DataLoader(LungDatasetImageOnly(df_test, CONFIG["BASE_PATH"], False),
                             batch_size=CONFIG["BATCH_SIZE_VAL"], shuffle=False, num_workers=CONFIG["NUM_WORKERS"])

        model = ImageOnlySurv().to(DEVICE)
        for p in model.backbone.parameters(): p.requires_grad = False
        opt = torch.optim.Adam(model.parameters(), lr=CONFIG["LR_HEAD"])

        best_cidx, wait = -1, 0
        model_path = os.path.join(OUTPUT_DIR, f"best_model_fold{fold}.pth")

        for epoch in range(CONFIG["EPOCHS"]):
            if epoch == CONFIG["WARMUP_EPOCHS"]:
                for p in model.backbone.parameters(): p.requires_grad = True

            model.train()
            for vol, time, event in train_ld:
                vol, time, event = vol.to(DEVICE), time.to(DEVICE), event.to(DEVICE)
                if event.sum() == 0: continue
                opt.zero_grad()
                loss = cox_loss(model(vol), time, event)
                loss.backward()
                opt.step()

            # Validation
            risks_v, times_v, events_v = validate(model, val_ld)
            cidx = concordance_index_censored(events_v.astype(bool), times_v, risks_v)[0]

            if epoch % 10 == 0:
                print(f"Epoch {epoch:03d} | Val C-index: {cidx:.4f}")

            if cidx > best_cidx:
                best_cidx = cidx
                wait = 0
                torch.save(model.state_dict(), model_path)
            else:
                wait += 1
                if wait >= CONFIG["PATIENCE"]: break

       
        model.load_state_dict(torch.load(model_path))
        risks_t, times_t, events_t = validate(model, train_ld)
        risks_test, times_test, events_test = validate(model, test_ld)

        fold_cindex = concordance_index_censored(events_test.astype(bool), times_test, risks_test)[0]
        cindex_folds.append(fold_cindex)

        # IBS
        breslow = BreslowEstimator().fit(risks_t, events_t.astype(bool), times_t)
        surv_funcs = breslow.get_survival_function(risks_test)
        t_min, t_max = max(times_test.min(), times_t.min()), min(times_test.max(), times_t.max())

        if t_max - t_min > 1:
            grid = np.linspace(t_min, t_max, 100, endpoint=False)
            preds = np.vstack([fn(grid) for fn in surv_funcs])
            fold_ibs = integrated_brier_score(Surv.from_arrays(events_t.astype(bool), times_t),
                                              Surv.from_arrays(events_test.astype(bool), times_test),
                                              preds, grid)
            ibs_folds.append(fold_ibs)
        else:
            ibs_folds.append(np.nan)

        
        all_risks.append(risks_test)
        all_times.append(times_test)
        all_events.append(events_test)
        print(f"FOLD {fold} FINAL -> C-Index: {fold_cindex:.4f}")

    
    c_m, c_s, c_ci = get_stats(cindex_folds)
    valid_ibs = [x for x in ibs_folds if not np.isnan(x)]
    i_m, i_s, i_ci = get_stats(valid_ibs) if valid_ibs else (0,0,(0,0))

    # Kaplan-Meier Global
    risks_total = np.concatenate(all_risks)
    times_total = np.concatenate(all_times)
    events_total = np.concatenate(all_events)
    p_val = logrank_test(times_total[risks_total >= np.median(risks_total)],
                         times_total[risks_total < np.median(risks_total)],
                         events_total[risks_total >= np.median(risks_total)],
                         events_total[risks_total < np.median(risks_total)]).p_value

    print("\n" + "="*60)
    print(f"C-Index: {c_m:.4f} ± {c_s:.4f} | IC95%: [{c_ci[0]:.4f}, {c_ci[1]:.4f}]")
    print(f"IBS:     {i_m:.4f} ± {i_s:.4f} | IC95%: [{i_ci[0]:.4f}, {i_ci[1]:.4f}]")
    print(f"Log-rank p-value: {p_val:.6f}")
    print("="*60)

    # JSON
    metrics_summary = {
        "model_name": "Modelo 3: Unimodal ResNet3D",
        "final_stats": {
            "cindex_mean": float(c_m), "cindex_std": float(c_s), "cindex_ci_95": [float(c_ci[0]), float(c_ci[1])],
            "ibs_mean": float(i_m), "logrank_p_value": float(p_val)
        },
        "config": {k: str(v) for k, v in CONFIG.items()}
    }
    with open(os.path.join(OUTPUT_DIR, "m3_metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=4)

    # Plot KM
    plt.figure(figsize=(8, 6))
    kmf = KaplanMeierFitter()
    
    thr = np.median(risks_total)
    high = risks_total >= thr
    low = risks_total < thr

    kmf.fit(times_total[high]/365.25, events_total[high], label=f"High Risk (n={sum(high)})").plot_survival_function()
    kmf.fit(times_total[low]/365.25, events_total[low], label=f"Low Risk (n={sum(low)})").plot_survival_function()
    plt.grid(alpha=0.3);
    plt.tight_layout();
    plt.savefig(os.path.join(OUTPUT_DIR, "km_model3.png"));
    plt.show()

if __name__ == "__main__":
    main()